# 05 E-backtesting

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from src.risk_measures import fhs_var_es
from src.pipeline import build_models

m = build_models()
loss = m['loss']
sigma_ewma = m['sigma_ewma']
hs_main, ewma_main, fhs_main = m['hs_main'], m['ewma_main'], m['fhs_main']
c = m['common_main']

## E-values

In [ ]:
from src.backtests import e_values

e_hs   = e_values(loss.loc[c], hs_main['var'].loc[c],   hs_main['es'].loc[c])
e_ewma = e_values(loss.loc[c], ewma_main['VaR'].loc[c], ewma_main['ES'].loc[c])
e_fhs  = e_values(loss.loc[c], fhs_main['VaR'].loc[c],  fhs_main['ES'].loc[c])

print(pd.Series({'HS': e_hs.mean(), 'EWMA': e_ewma.mean(), 'FHS': e_fhs.mean()}))
print((e_fhs > 0).sum())   

In [ ]:
print((fhs_main['ES'] - fhs_main['VaR']).loc[c].describe())   # how small does the gap get?
print(e_fhs.nlargest(5))                                        # are a few days driving the mean?

In [ ]:
fhs_500 = fhs_var_es(loss, sigma_ewma, window=500)

c2 = fhs_500['VaR'].dropna().index          # days where the 500-day model has a forecast
print(len(c2))                              # should be about 2,422

e_250 = e_values(loss.loc[c2], fhs_main['VaR'].loc[c2], fhs_main['ES'].loc[c2])
e_500 = e_values(loss.loc[c2], fhs_500['VaR'].loc[c2],  fhs_500['ES'].loc[c2])

print(pd.Series({'FHS 250': e_250.mean(), 'FHS 500': e_500.mean()}))
print(e_500.nlargest(3))

## Fixed λ

In [ ]:
from src.backtests import e_process

for name, e in {'HS': e_hs, 'EWMA': e_ewma, 'FHS': e_fhs}.items():
    w, d = e_process(e, lam=0.02)
    print(name, round(w.max(), 2), d.date() if type(d) == pd.Timestamp else None)

In [ ]:
plt.figure()
for name, e in {'HS': e_hs, 'EWMA': e_ewma, 'FHS': e_fhs}.items():
    w, d = e_process(e, lam=0.02)
    plt.plot(np.log10(w), label=name)
    if d is not None:
        plt.scatter(d, np.log10(w.loc[d]), marker='x', color='black', zorder=5)   # rejection point
plt.axhline(np.log10(20), color='black', linestyle='dotted', label='Rejection threshold (α = 5%)')
plt.title(r'E-backtest wealth ($\lambda = 0.02$)')
plt.xlabel('Date')
plt.ylabel(r'$\log_{10}$ wealth')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.savefig(f'{ROOT}\\results\\e_backtest_wealth.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for lam in [0.01, 0.02, 0.05]:
    for name, e in {'HS': e_hs, 'EWMA': e_ewma, 'FHS': e_fhs}.items():
        w, d = e_process(e, lam=lam)
        print(lam, name, d)

## Adaptive λ

In [ ]:
from src.backtests import e_process_adaptive

results_adaptive = {}
for name, e in {'HS': e_hs, 'EWMA': e_ewma, 'FHS': e_fhs}.items():
    w, d, lam = e_process_adaptive(e)
    results_adaptive[name] = (w, d, lam)
    print(f"{name}: rejected on {d}, max wealth {w.max():.2f}, mean λ {lam.mean():.3f}")

In [ ]:
w5, d5, lam5 = e_process_adaptive(e_500)                      
w2, d2, lam2 = e_process_adaptive(e_250)                       
print("FHS-250:", d2, round(w2.max(), 2))
print("FHS-500:", d5, round(w5.max(), 2))

In [ ]:
plt.figure(figsize=(9, 5))

lines = {
    'HS': results_adaptive['HS'][:2],
    'EWMA': results_adaptive['EWMA'][:2],
    'FHS (250-day)': results_adaptive['FHS'][:2],
    'FHS (500-day, from 2017)': (w5, d5)}

for name, (w, d) in lines.items():
    plt.plot(np.log10(w), label=name)
    if d is not None:
        plt.scatter(d, np.log10(w.loc[d]), marker='x', color='black', zorder=5)

plt.axhline(np.log10(20), color='black', linestyle='dotted', label='Rejection threshold (α = 5%)')
plt.title(r'E-backtest wealth (adaptive $\lambda$)')
plt.xlabel('Date')
plt.ylabel(r'$\log_{10}$ wealth')
plt.legend(loc='upper left')
plt.savefig(f'{ROOT}\\results\\e_backtest_wealth_adaptive.png', dpi=150, bbox_inches='tight')
plt.show()